In [1]:
import pulp

# --- Sets ---
hubs = ['CVG', 'AFW']
focus_cities = ['Leipzig', 'Hyderabad', 'San_Bernardino']
centers = {
    'Paris': 6500, 'Cologne': 640, 'Hanover': 180, 'Bangalore': 9100, 'Coimbatore': 570, 'Delhi': 19000, 
    'Mumbai': 14800, 'Cagliari': 90, 'Catania': 185, 'Milan': 800, 'Rome': 1700, 'Katowice': 170, 
    'Barcelona': 2800, 'Madrid': 3700, 'Castle_Donington': 30, 'London': 6700, 'Mobile': 190,
    'Anchorage': 175, 'Fairbanks': 38, 'Phoenix': 2400, 'Los_Angeles': 7200,
    'Ontario': 100, 'Riverside': 1200, 'Sacramento': 1100, 'San_Francisco': 1900, 'Stockton': 240,
    'Denver': 1500, 'Hartford': 540, 'Miami': 3400, 'Lakeland': 185, 'Tampa': 1600, 'Atlanta': 3000,
    'Honolulu': 500, 'Kahului': 16, 'Kona': 63, 'Chicago': 5100, 'Rockford': 172, 'Fort_Wayne': 200,
    'South_Bend': 173, 'Des_Moines': 300, 'Wichita': 290, 'New_Orleans': 550, 'Baltimore': 1300,
    'Minneapolis': 1700, 'Kansas_City': 975, 'St_Louis': 1200, 'Omaha': 480, 'Manchester': 100,
    'Albuquerque': 450, 'New_York': 11200, 'Charlotte': 900, 'Toledo': 290, 'Wilmington': 150,
    'Portland': 1200, 'Allentown': 420, 'Pittsburgh': 1000, 'San_Juan': 1100, 'Nashville': 650,
    'Austin': 975, 'Dallas': 3300, 'Houston': 3300, 'San_Antonio': 1100, 'Richmond': 600,
    'Seattle_Tacoma': 2000, 'Spokane': 260
}

# --- Parameters ---
hub_capacities = {'CVG': 95650, 'AFW': 44350}
focus_capacities = {'Leipzig': 85000, 'Hyderabad': 19000, 'San_Bernardino': 36000}

# Shipping costs between valid nodes (subset for demo)
costs = {
    ('CVG', 'Paris'): 1.6,
    ('CVG', 'Mobile'): 0.5,
    ('AFW', 'Mobile'): 0.5,
    ('Leipzig', 'Paris'): 0.5,
    ('San_Bernardino', 'Mobile'): 0.5,
    ('CVG', 'Leipzig'): 1.5,
    ('AFW', 'San_Bernardino'): 0.5
    # Add more tuples here for full network
}

# --- Initialize Model ---
model = pulp.LpProblem("Amazon_Air_Distribution_Optimization", pulp.LpMinimize)

# --- Decision Variables ---
x = pulp.LpVariable.dicts("ship", costs, lowBound=0)

# --- Objective Function ---
model += pulp.lpSum([x[i, j] * costs[i, j] for (i, j) in costs])

# --- Constraints ---

# Center demand (subset for demo)
for center, demand in centers.items():
    incoming = [x[i, j] for (i, j) in costs if j == center]
    if incoming:
        model += pulp.lpSum(incoming) == demand, f"demand_{center}"

# Hub capacity
for hub in hubs:
    outgoing = [x[i, j] for (i, j) in costs if i == hub]
    model += pulp.lpSum(outgoing) <= hub_capacities[hub], f"capacity_{hub}"

# Focus city capacity
for focus in focus_cities:
    outgoing = [x[i, j] for (i, j) in costs if i == focus]
    model += pulp.lpSum(outgoing) <= focus_capacities[focus], f"capacity_{focus}"

# --- Solve ---
model.solve()

# --- Output ---
print(f"Status: {pulp.LpStatus[model.status]}")
print(f"Total cost: {pulp.value(model.objective)}")
for var in model.variables():
    if var.varValue > 0:
        print(f"{var.name} = {var.varValue}")


Status: Optimal
Total cost: 3345.0
ship_('AFW',_'Mobile') = 190.0
ship_('Leipzig',_'Paris') = 6500.0
